In [1]:
import pandas as pd
import gc
import os
import glob

In [2]:
folder_path = r"C:\Users\samuelbarroso\Downloads\Dados\SCR data antiga\planilha_2019"
output_file = r"C:\Users\samuelbarroso\Downloads\Dados\SCR data antiga\planilha_2019\planilha_2019_agrupada_unpivot"
files = glob.glob(os.path.join(folder_path, "planilha_2019*.csv"))
files = [f for f in files if "completa" not in f and "agrupada" not in f]
files.sort()

value_vars = [
    "numero_de_operacoes",
    "a_vencer_ate_90_dias",
    "a_vencer_de_91_ate_360_dias",
    "a_vencer_de_361_ate_1080_dias",
    "a_vencer_de_1081_ate_1800_dias",
    "a_vencer_de_1801_ate_5400_dias",
    "a_vencer_acima_de_5400_dias",
    "vencido_acima_de_15_dias",
    "carteira_ativa",
    "carteira_inadimplida_arrastada",
    "ativo_problematico"
]

id_vars = [
    "data_base", "uf", "tcb", "sr", "cliente", "ocupacao", 
    "cnae_secao", "cnae_subclasse", "porte", "modalidade", "origem", "indexador"
]


In [34]:
folder_path = r"C:\Users\samuelbarroso\Downloads\Dados\SCR data antiga\planilha_2019\planilha_201901.csv"
chunk = pd.read_csv(filepath, sep=";",encoding="utf-8-sig", dtype=str)

In [35]:
chunk.head()

,data_base,uf,tcb,sr,cliente,ocupacao,cnae_secao,cnae_subclasse,porte,modalidade,...,a_vencer_ate_90_dias,a_vencer_de_91_ate_360_dias,a_vencer_de_361_ate_1080_dias,a_vencer_de_1081_ate_1800_dias,a_vencer_de_1801_ate_5400_dias,a_vencer_acima_de_5400_dias,vencido_acima_de_15_dias,carteira_ativa,carteira_inadimplida_arrastada,ativo_problematico
0,2019-02-28,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,...,"3785007,87","815353,45","60658,54","1973,55","3450,39","14864,66","132737,29","4814045,75","110089,97","200249,08"
1,2019-02-28,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,...,"0,00","0,00","0,00","0,00","0,00","0,00","788,62","788,62","788,62","788,62"
2,2019-02-28,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Empréstimo com consignação em folha,...,"1541793,84","4677336,62","9333160,56","6090233,79","4388607,10","486,80","18750,71","26050369,42","9813,59","9813,59"
3,2019-02-28,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Empréstimo com consignação em folha,...,"0,00","0,00","0,00","0,00","0,00","0,00","140105,71","140105,71","140105,71","140105,71"
4,2019-02-28,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Empréstimo sem consignação em folha,...,"322083,26","1029176,90","1274017,24","517550,90","270533,94","38613,19","15519,96","3467495,39","26193,66","134692,37"


In [15]:
folder_path = r"C:\Users\samuelbarroso\Downloads\Dados\SCR data antiga\planilha_2019\planilha_201901.csv"
chunk = pd.read_csv(filepath, sep=";",encoding="utf-8-sig", dtype=str,decimal=",")

for col in id_vars:
        if col in chunk.columns:
            chunk[col] = chunk[col].str.strip()
if 'numero_de_operacoes' in chunk.columns:
        chunk['numero_de_operacoes'] = chunk['numero_de_operacoes'].astype(str).str.replace('<= 15', '7').str.replace('<=15', '7').str.strip()
        chunk['numero_de_operacoes'] = pd.to_numeric(chunk['numero_de_operacoes'], errors='coerce').fillna(0).astype('float64')
cols_to_melt = [c for c in value_vars if c in chunk.columns]
chunk[cols_to_melt] = (
    chunk[cols_to_melt]
    .replace(",", ".", regex=True)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype("float64")
)
melted = chunk.melt(id_vars=id_vars, value_vars=cols_to_melt, var_name='Atributo', value_name='Valor')
    
grouped = melted.groupby(id_vars + ['Atributo'], as_index=False)['Valor'].sum()
# Valor numérico
grouped['Valor'] = pd.to_numeric(grouped['Valor'], errors='coerce').fillna(0)

In [16]:
grouped.head()

,data_base,uf,tcb,sr,cliente,ocupacao,cnae_secao,cnae_subclasse,porte,modalidade,origem,indexador,Atributo,Valor
0,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,Sem destinação específica,Prefixado,a_vencer_acima_de_5400_dias,3048.97
1,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,Sem destinação específica,Prefixado,a_vencer_ate_90_dias,5098923.32
2,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,Sem destinação específica,Prefixado,a_vencer_de_1081_ate_1800_dias,6952.09
3,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,Sem destinação específica,Prefixado,a_vencer_de_1801_ate_5400_dias,16032.60
4,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,Sem destinação específica,Prefixado,a_vencer_de_361_ate_1080_dias,78952.67


In [25]:
len(chunk)
#len(grouped)

649271

In [21]:

grouped = grouped[grouped['Valor'] != 0]

# data_base como data
grouped['data_base'] = pd.to_datetime(grouped['data_base'], errors='coerce')

# todas as outras colunas como string
cols_str = [c for c in grouped.columns if c not in ['Valor', 'data_base']]
grouped[cols_str] = grouped[cols_str].astype(str)

In [22]:
len(grouped)

647625

In [14]:
grouped.dtypes

data_base         object
uf                object
tcb               object
sr                object
cliente           object
ocupacao          object
cnae_secao        object
cnae_subclasse    object
porte             object
modalidade        object
origem            object
indexador         object
Atributo          object
Valor             object
dtype: object

In [6]:
melted.head()

,data_base,uf,tcb,sr,cliente,ocupacao,cnae_secao,cnae_subclasse,porte,modalidade,origem,indexador,Atributo,Valor
0,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Cartão de crédito,Sem destinação específica,Prefixado,numero_de_operacoes,1070.0
1,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Empréstimo com consignação em folha,Sem destinação específica,Prefixado,numero_de_operacoes,847.0
2,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Empréstimo com consignação em folha,Sem destinação específica,Pós-fixado,numero_de_operacoes,7.0
3,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Empréstimo sem consignação em folha,Sem destinação específica,Prefixado,numero_de_operacoes,197.0
4,2019-12-31,AC,Bancário,S1,PF,PF - Aposentado/pensionista,-,-,PF - Acima de 20 salários mínimos,PF - Habitacional,Com destinação específica,Pós-fixado,numero_de_operacoes,35.0


In [17]:
for file_idx, filepath in enumerate(files):
    print(file_idx, filepath )
    chunk = pd.read_csv(filepath, sep=";",encoding="utf-8-sig", dtype=str)
    #chunk.rename(columns=lambda x: x.replace('ï»¿', '').replace('\ufeff', '').strip(), inplace=True)
    for col in id_vars:
            if col in chunk.columns:
                chunk[col] = chunk[col].str.strip()
    if 'numero_de_operacoes' in chunk.columns:
            chunk['numero_de_operacoes'] = chunk['numero_de_operacoes'].astype(str).str.replace('<= 15', '7').str.replace('<=15', '7').str.strip()
            chunk['numero_de_operacoes'] = pd.to_numeric(chunk['numero_de_operacoes'], errors='coerce').fillna(0).astype('float64')
    cols_to_melt = [c for c in value_vars if c in chunk.columns]
    chunk[cols_to_melt] = (
    chunk[cols_to_melt]
    .replace(",", ".", regex=True)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype("float64")
    )
    melted = chunk.melt(id_vars=id_vars, value_vars=cols_to_melt, var_name='Atributo', value_name='Valor')
        
    grouped = melted.groupby(id_vars + ['Atributo'], as_index=False)['Valor'].sum()
    # Valor numérico
    grouped['Valor'] = pd.to_numeric(grouped['Valor'], errors='coerce').fillna(0)
    

    # data_base como data
    grouped['data_base'] = pd.to_datetime(grouped['data_base'], errors='coerce')

    # todas as outras colunas como string
    cols_str = [c for c in grouped.columns if c not in ['Valor', 'data_base']]
    grouped[cols_str] = grouped[cols_str].astype(str)

    grouped.to_parquet(output_file+str(file_idx)+'.parquet')  
    grouped = grouped[grouped['Valor'] != 0]

    chunk.to_csv(output_file+str(file_idx)+'.csv',  sep=';',encoding="utf-8-sig", index=False)  
    grouped.to_csv(output_file+str(file_idx)+'grouped.csv',  sep=';',encoding="utf-8-sig", index=False) 
     

0 C:\Users\samuelbarroso\Downloads\Dados\SCR data antiga\planilha_2019\planilha_201901.csv
1 C:\Users\samuelbarroso\Downloads\Dados\SCR data antiga\planilha_2019\planilha_201902.csv


KeyboardInterrupt: 

In [18]:
len(grouped)

7123875

In [11]:
!pip install charmap

ERROR: Could not find a version that satisfies the requirement charmap (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for charmap


In [12]:
python.exe -m pip install --upgrade pip

SyntaxError: invalid syntax (842801469.py, line 1)